# Using digitised values

Digitising is a means: the values are for something. Two examples with the paper's figures, each checked against
what the paper's own data give.

## Figure 1b, worked out again from the line of Figure 1a

Figure 1a draws the number of births in each month from 1955 to 2014; Figure 1b shows, for each month of the year,
how much fewer births there were from 1975 on than before. So Figure 1b can be worked out from Figure 1a - from
its *picture*. Read the blue line, take its value in the middle of each month, and compare the two eras.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import trace_digitiser as td

EX = "/data/examples/hearn-whitmore-2026"
line = td.digitise(f"{EX}/figure-1a.png", trace_colours=["#4682b4"], exclude=[(120, 15, 262, 58)],
                   write_outputs=False)
d = line.trace_data
months = pd.read_csv(f"{EX}/figure-1a.known.csv", parse_dates=["date"])   # (only its dates are used here)
months["births_read"] = np.interp(months.decimal_year, d.x_value, d.mean_estimate)
months["month"] = months.date.dt.month
before = months[months.date.dt.year <= 1974].groupby("month").births_read.mean()
after = months[months.date.dt.year >= 1975].groupby("month").births_read.mean()
read = 100 * (after - before) / before
paper = pd.read_csv(f"{EX}/figure-1b.known.csv")
table = pd.DataFrame({"month": paper.label, "from the picture of 1a": read.round(1).values,
                      "Figure 1b": paper.percent_difference.round(1).values})
table

In [ ]:
x = np.arange(12)
plt.figure(figsize=(7, 3.5))
plt.bar(x - 0.2, table["Figure 1b"], width=0.4, label="Figure 1b (the data)")
plt.bar(x + 0.2, table["from the picture of 1a"], width=0.4, label="worked out from the picture of Figure 1a")
plt.xticks(x, table.month)
plt.ylabel("% difference, 1975- against 1955-1974")
plt.legend(fontsize=8)
plt.show()
print("largest difference:", (table["Figure 1b"] - table["from the picture of 1a"]).abs().max().round(2), "points")

Within a point or two everywhere: the shape of the result - fewest births lost in the autumn, most in the
spring - comes out of the picture alone.

## The peak of the birthogram, year by year

Figure 2 shows each year as a row: its months' births scaled from 0 (the year's fewest) to 1 (its most). Read with
24 bins - two years of months, as it is drawn - the first twelve bins of each row are its own year. The month with
the most births each year moves in the mid-1970s, which is what the paper is about.

In [ ]:
rows = td.digitise_rows(f"{EX}/figure-2.png", row_values=list(range(1955, 2014)), bins=24,
                       row_spacing_value=1.1, write_outputs=False).data
own = rows[rows.bin <= 12].copy()
peak = own.loc[own.groupby("row_value").value.idxmax(), ["row_value", "bin"]].astype(int)
peak.columns = ["year", "peak month"]
known = pd.read_csv(f"{EX}/figure-2.known.csv")
k = known[known.column < 12]
known_peak = k.loc[k.groupby("row_year").normalised_births.idxmax(), ["row_year", "column"]]
peak["peak month (data)"] = known_peak.column.values + 1
print("the same peak month in", (peak["peak month"] == peak["peak month (data)"]).sum(), "of", len(peak), "years")
plt.figure(figsize=(7, 3))
plt.plot(peak.year, peak["peak month"], "o-", label="read from the picture")
plt.plot(peak.year, peak["peak month (data)"], "x", label="the data")
plt.yticks(range(1, 13))
plt.ylabel("month with most births")
plt.legend(fontsize=8)
plt.show()